# Launch Reception Predictor & Sentiment Forecaster

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Analyze predict** using Snowflake Cortex AI
2. **Build production pipelines** with SQL and SENTIMENT()
3. **Create data structures** for Time-series forecasting
4. **Implement business logic** for communications intelligence
5. **Generar insights** with window functions and aggregations
6. **Build interactive dashboards** for stakeholder reporting

---

## What You'll Build

A **launch reception predictor & sentiment forecaster** that provides:
- Predict sentiment trajectory and reception for new product launches using statistical analysis
- Automated data collection and processing
- Real-time analysis and insights
- Interactive visualization dashboard
- Production-ready SQL for scale

---

## Technical Requirements

- **Snowflake account** with Cortex AI enabled
- **Approximately 12-15 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features

- `AI_SENTIMENT()` - Primary AI function
- Statistical trend analysis - Prediction modeling
- Window functions - Time-series analysis
- Aggregations - Summary statistics
- CTEs - Complex query logic
- `CASE` statements - Classification

Let's begin!

---

## Paso 1: Configuración del Entorno

### Qué Vamos a Crear

- **Database**: `LAUNCH_PREDICTOR_DB`
- **Schema**: `PUBLIC`
- **Warehouse**: `COMPUTE_WH`

### Business Challenge

Predict sentiment trajectory and reception for new product launches using ML

### Why This Matters

Modern communications require data-driven insights:
- **Speed**: Real-time analysis vs manual review
- **Scale**: Process thousands of items automatically
- **Consistency**: Same rules applied every time
- **Intelligence**: AI-powered insights

In [ ]:
-- Create environment
CREATE DATABASE IF NOT EXISTS LAUNCH_PREDICTOR_DB;
CREATE SCHEMA IF NOT EXISTS LAUNCH_PREDICTOR_DB.PUBLIC;
USE SCHEMA LAUNCH_PREDICTOR_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT 
    CURRENT_DATABASE() as database_name,
    CURRENT_SCHEMA() as schema_name,
    CURRENT_WAREHOUSE() as warehouse_name,
    'Launch Reception Predictor & Sentiment Forecaster Environment Ready!' as status;

---

## Paso 2: Define Data Structure

### Tables

1. **`launch_sentiment_history`** - Primary data source
2. **`sentiment_forecasts`** - Analysis results
3. **`launch_predictions`** - Aggregated insights

### Data Flow

1. **Ingest** → Collect data from sources
2. **Process** → Apply AI functions
3. **Analyze** → Calculate metrics
4. **Visualize** → Present insights

In [ ]:
-- Primary data table
CREATE OR REPLACE TABLE launch_sentiment_history (
    record_id VARCHAR(50) PRIMARY KEY,
    source VARCHAR(100),
    content TEXT,
    created_date DATE,
    metadata VARIANT,
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- Analysis results table
CREATE OR REPLACE TABLE sentiment_forecasts (
    analysis_id VARCHAR(50) PRIMARY KEY,
    record_id VARCHAR(50),
    analysis_result FLOAT,
    category VARCHAR(50),
    confidence FLOAT,
    analyzed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    FOREIGN KEY (record_id) REFERENCES launch_sentiment_history(record_id)
);

-- Aggregated insights table
CREATE OR REPLACE TABLE launch_predictions (
    insight_id VARCHAR(50) PRIMARY KEY,
    date_period DATE,
    metric_name VARCHAR(100),
    metric_value FLOAT,
    trend VARCHAR(20),
    calculated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

SELECT 'Tables created successfully!' as status;

---

## Paso 3: Generar Datos Sintéticos Data

### Qué Vamos a Crear

- **1,000 records** for demonstration
- **Realistic content** matching use case
- **Last 30 days** of data
- **Multiple sources** and categories

### Synthetic Data Approach

Using Snowflake's `GENERATOR()` and `UNIFORM()` functions to create realistic test data.

In [ ]:
-- Generar datos sintéticos data
INSERT INTO launch_sentiment_history
WITH sources AS (
    SELECT * FROM (VALUES
        ('source_a'), ('source_b'), ('source_c'), ('source_d'), ('source_e')
    ) t(source)
),
content_templates AS (
    SELECT * FROM (VALUES
        ('Positive content example for Launch analysis'),
        ('Neutral content example for testing and validation purposes'),
        ('Negative content example showing issues and concerns'),
        ('Mixed sentiment content with both positive and negative aspects'),
        ('Technical content focusing on specific product features')
    ) t(template)
)
SELECT 
    'REC_' || LPAD(SEQ4()::VARCHAR, 8, '0') as record_id,
    s.source,
    REPLACE(ct.template, 'example', 'item ' || SEQ4()::VARCHAR) as content,
    DATEADD('day', -FLOOR(UNIFORM(1, 30, RANDOM())), CURRENT_DATE()) as created_date,
    OBJECT_CONSTRUCT(
        'category', ARRAY_CONSTRUCT('cat_a', 'cat_b', 'cat_c')[FLOOR(UNIFORM(0, 3, RANDOM()))],
        'priority', ARRAY_CONSTRUCT('low', 'medium', 'high')[FLOOR(UNIFORM(0, 3, RANDOM()))]
    ) as metadata,
    CURRENT_TIMESTAMP() as ingested_at
FROM TABLE(GENERATOR(ROWCOUNT => 1000)) g
CROSS JOIN sources s
CROSS JOIN content_templates ct
WHERE UNIFORM(0, 1, RANDOM()) < 0.2
QUALIFY ROW_NUMBER() OVER (ORDER BY UNIFORM(0, 1, RANDOM())) <= 1000;

SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT source) as sources,
    MIN(created_date) as earliest_date,
    MAX(created_date) as latest_date
FROM launch_sentiment_history;

---

## Paso 4: Apply AI Sentiment Analysis

### Using Cortex AI

Analyze launch feedback sentiment using AI:
- Real-time sentiment scoring
- Category classification
- Confidence scoring
- Production-ready performance

### Why Not FORECAST() Here?

Note: This cell analyzes existing sentiment. The actual launch prediction/forecasting happens in later cells using trend analysis and statistical methods.

In [ ]:
-- Analyze sentiment for launch reception history
-- Verified via Snowflake docs 2025-01: AI_SENTIMENT returns OBJECT
INSERT INTO sentiment_forecasts
SELECT 
    'ANL_' || LPAD(ROW_NUMBER() OVER (ORDER BY record_id), 8, '0') as analysis_id,
    record_id,
    -- Convert sentiment to numeric score for analysis_result column (Use Case 28 pattern)
    CASE AI_SENTIMENT(content)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 1.0
        WHEN 'neutral' THEN 0.0
        WHEN 'negative' THEN -1.0
        WHEN 'mixed' THEN -0.5
        ELSE -1.0
    END as analysis_result,
    -- Human-readable sentiment category
    CASE AI_SENTIMENT(content)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 'Positive'
        WHEN 'neutral' THEN 'Neutral'
        WHEN 'negative' THEN 'Negative'
        WHEN 'mixed' THEN 'Negative'
        ELSE 'Very Negative'
    END as category,
    0.95 as confidence,
    CURRENT_TIMESTAMP() as analyzed_at
FROM launch_sentiment_history;

-- View sentiment trends over time
SELECT 
    DATE_TRUNC('day', lsh.created_date) as launch_day,
    sf.category,
    COUNT(*) as feedback_count,
    ROUND(AVG(sf.analysis_result), 3) as avg_sentiment_score
FROM launch_sentiment_history lsh
JOIN sentiment_forecasts sf ON lsh.record_id = sf.record_id
GROUP BY DATE_TRUNC('day', lsh.created_date), sf.category
ORDER BY launch_day, avg_sentiment_score DESC;

---

## Paso 5: Calculate Aggregated Metrics

### Summary Statistics

Create daily/weekly aggregations for:
- Trend analysis
- Baseline calculations
- Anomaly detection

In [ ]:
-- Calculate daily metrics
INSERT INTO launch_predictions
SELECT 
    'INS_' || LPAD(ROW_NUMBER() OVER (ORDER BY created_date), 8, '0') as insight_id,
    created_date as date_period,
    'daily_average' as metric_name,
    ROUND(AVG(a.analysis_result), 3) as metric_value,
    CASE 
        WHEN AVG(a.analysis_result) > LAG(AVG(a.analysis_result)) OVER (ORDER BY created_date) THEN 'up'
        WHEN AVG(a.analysis_result) < LAG(AVG(a.analysis_result)) OVER (ORDER BY created_date) THEN 'down'
        ELSE 'stable'
    END as trend,
    CURRENT_TIMESTAMP() as calculated_at
FROM launch_sentiment_history r
INNER JOIN sentiment_forecasts a ON r.record_id = a.record_id
GROUP BY created_date;

SELECT * FROM launch_predictions ORDER BY date_period DESC LIMIT 10;

---

## Paso 6: Sentiment Distribution by Launch Phase

### Qué Vamos a Hacer

Analyze how sentiment varies across different **launch phases**: pre-launch, launch week, early adoption, and growth.

### Why This Matters

- **Phase-specific patterns**: Each launch phase has different sentiment dynamics
- **Predictive insights**: Historical patterns help forecast future launches
- **Risk identification**: Know when sentiment typically dips

### Key Concepts

**Launch Phases**:
- **Pre-launch** (-30 to -1 days): Anticipation and speculation
- **Launch week** (0-7 days): Initial reactions
- **Early adoption** (8-90 days): Real-world experience emerges
- **Growth** (91+ days): Established sentiment patterns

Uses `CASE` statements and window functions for phase analysis.

In [ ]:
-- Analyze sentiment distribution by launch phase
SELECT 
    CASE 
        WHEN DATEDIFF('day', '2017-12-05'::DATE, lsh.created_date) < 0 THEN 'pre_launch'
        WHEN DATEDIFF('day', '2017-12-05'::DATE, lsh.created_date) BETWEEN 0 AND 7 THEN 'launch_week'
        WHEN DATEDIFF('day', '2017-12-05'::DATE, lsh.created_date) BETWEEN 8 AND 90 THEN 'early_adoption'
        ELSE 'growth'
    END as launch_phase,
    COUNT(DISTINCT lsh.created_date) as days_with_feedback,
    COUNT(*) as total_feedback,
    ROUND(AVG(sf.analysis_result), 3) as phase_avg_sentiment,
    ROUND(STDDEV(sf.analysis_result), 3) as sentiment_volatility,
    ROUND(MIN(sf.analysis_result), 3) as lowest_sentiment,
    ROUND(MAX(sf.analysis_result), 3) as highest_sentiment,
    ROUND(100.0 * COUNT(CASE WHEN sf.analysis_result >= 0.5 THEN 1 END) / COUNT(*), 1) as positive_pct
FROM launch_sentiment_history lsh
INNER JOIN sentiment_forecasts sf ON lsh.record_id = sf.record_id
GROUP BY launch_phase
ORDER BY 
    CASE launch_phase
        WHEN 'pre_launch' THEN 1
        WHEN 'launch_week' THEN 2
        WHEN 'early_adoption' THEN 3
        WHEN 'growth' THEN 4
    END;

---

## Paso 7: Time-Series Sentiment Trending

### Qué Vamos a Hacer

Calculate **rolling averages** and **momentum** to identify sentiment trends over time.

### Why This Matters

- **Trend detection**: Spot upward or downward momentum early
- **Smooth volatility**: Rolling averages reduce daily noise
- **Forecast preparation**: Trends inform prediction models

### Key Functions

- **`AVG() OVER (... ROWS BETWEEN ...)`**: Rolling window average
- **`LAG()`**: Compare to previous periods
- **Momentum calculation**: Current vs 7-day average

In [ ]:
-- Calculate rolling averages and sentiment momentum
WITH daily_avg AS (
    SELECT 
        lsh.created_date,
        AVG(sf.analysis_result) as daily_sentiment,
        COUNT(*) as feedback_volume
    FROM launch_sentiment_history lsh
    INNER JOIN sentiment_forecasts sf ON lsh.record_id = sf.record_id
    GROUP BY lsh.created_date
)
SELECT 
    created_date,
    DATEDIFF('day', '2017-12-05'::DATE, created_date) as days_since_launch,
    ROUND(daily_sentiment, 3) as daily_sentiment,
    ROUND(AVG(daily_sentiment) OVER (ORDER BY created_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 3) as rolling_7day_avg,
    ROUND(AVG(daily_sentiment) OVER (ORDER BY created_date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW), 3) as rolling_30day_avg,
    ROUND(daily_sentiment - LAG(daily_sentiment, 7) OVER (ORDER BY created_date), 3) as sentiment_change_7d,
    CASE 
        WHEN daily_sentiment > AVG(daily_sentiment) OVER (ORDER BY created_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) THEN 'IMPROVING'
        WHEN daily_sentiment < AVG(daily_sentiment) OVER (ORDER BY created_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) THEN 'DECLINING'
        ELSE 'STABLE'
    END as momentum,
    feedback_volume
FROM daily_avg
ORDER BY created_date DESC
LIMIT 30;

---

## Paso 8: Source Performance Analysis

### Qué Vamos a Hacer

Compare sentiment across different **feedback sources**: social media, news, medical journals, patient forums.

### Why This Matters

- **Source credibility**: Medical journals vs social media patterns
- **Channel strategy**: Know where to focus communications
- **Early indicators**: Which sources lead sentiment shifts?

### Analysis Approach

- Group by dominant source per day
- Calculate average sentiment by source
- Identify volume vs sentiment correlation

In [ ]:
-- Analyze sentiment by feedback source
SELECT 
    lsh.source,
    COUNT(*) as total_feedback,
    COUNT(DISTINCT lsh.created_date) as active_days,
    ROUND(AVG(sf.analysis_result), 3) as overall_avg_sentiment,
    ROUND(STDDEV(sf.analysis_result), 3) as sentiment_consistency,
    ROUND(MIN(sf.analysis_result), 3) as worst_sentiment,
    ROUND(MAX(sf.analysis_result), 3) as best_sentiment,
    COUNT(CASE WHEN sf.analysis_result >= 0.5 THEN 1 END) as positive_count,
    ROUND(100.0 * COUNT(CASE WHEN sf.analysis_result >= 0.5 THEN 1 END) / COUNT(*), 1) as positive_pct,
    RANK() OVER (ORDER BY AVG(sf.analysis_result) DESC) as sentiment_rank,
    RANK() OVER (ORDER BY COUNT(*) DESC) as volume_rank
FROM launch_sentiment_history lsh
INNER JOIN sentiment_forecasts sf ON lsh.record_id = sf.record_id
GROUP BY lsh.source
ORDER BY overall_avg_sentiment DESC;

---

## Paso 9: Stakeholder Sentiment Patterns

### Qué Vamos a Hacer

Analyze sentiment differences across **stakeholder types**: patients, physicians, media, researchers.

### Why This Matters

- **Target messaging**: Different stakeholders need different approaches
- **Priority identification**: Focus on physician sentiment for clinical adoption
- **Pattern recognition**: Patients vs physicians may differ in timing

### Stakeholder Types

- **Patients**: Direct experience with medication
- **Physicians**: Clinical efficacy and safety focus
- **Media**: Public perception and news coverage
- **Researchers**: Scientific evidence and study results

In [ ]:
-- Analyze sentiment patterns by category and priority
SELECT 
    lsh.metadata:category::STRING as category,
    lsh.metadata:priority::STRING as priority,
    COUNT(*) as feedback_count,
    ROUND(AVG(sf.analysis_result), 3) as avg_sentiment,
    ROUND(AVG(sf.confidence), 3) as avg_confidence,
    ROUND(MIN(sf.analysis_result), 3) as lowest_sentiment,
    ROUND(MAX(sf.analysis_result), 3) as highest_sentiment,
    COUNT(CASE WHEN sf.analysis_result >= 0.5 THEN 1 END) as positive_count,
    COUNT(CASE WHEN sf.analysis_result <= -0.5 THEN 1 END) as negative_count,
    RANK() OVER (PARTITION BY lsh.metadata:category::STRING ORDER BY AVG(sf.analysis_result) DESC) as priority_rank,
    CASE 
        WHEN AVG(sf.analysis_result) >= 0.5 THEN '✅ Positive'
        WHEN AVG(sf.analysis_result) <= -0.5 THEN '❌ Negative'
        ELSE '⚠️ Neutral'
    END as sentiment_flag
FROM launch_sentiment_history lsh
INNER JOIN sentiment_forecasts sf ON lsh.record_id = sf.record_id
GROUP BY lsh.metadata:category::STRING, lsh.metadata:priority::STRING
ORDER BY category, avg_sentiment DESC;

---

## Paso 10: Identify Predictive Indicators

### Qué Vamos a Hacer

Find **leading indicators** that predict sentiment changes before they happen.

### Why This Matters

- **Early warning system**: Detect problems before they escalate
- **Proactive response**: Time to prepare communications strategy
- **Forecast input**: Key variables for ML models

### Indicators to Analyze

- **Volume spikes**: Sudden increase in feedback
- **Source shifts**: Changes in dominant channel
- **Day-of-week patterns**: Weekday vs weekend sentiment
- **Correlation with events**: Product announcements, studies released

In [ ]:
-- Identify leading indicators for sentiment changes
WITH daily_metrics AS (
    SELECT 
        lsh.created_date,
        COUNT(*) as feedback_volume,
        AVG(sf.analysis_result) as daily_sentiment,
        COUNT(CASE WHEN sf.analysis_result >= 0.5 THEN 1 END) as positive_count,
        COUNT(CASE WHEN sf.analysis_result <= -0.5 THEN 1 END) as negative_count
    FROM launch_sentiment_history lsh
    INNER JOIN sentiment_forecasts sf ON lsh.record_id = sf.record_id
    GROUP BY lsh.created_date
),
indicators AS (
    SELECT 
        created_date,
        feedback_volume,
        daily_sentiment,
        AVG(feedback_volume) OVER (ORDER BY created_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as avg_volume_7d,
        LAG(daily_sentiment, 3) OVER (ORDER BY created_date) as sentiment_3d_ago,
        CASE 
            WHEN feedback_volume > 2 * AVG(feedback_volume) OVER (ORDER BY created_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) THEN TRUE
            ELSE FALSE
        END as volume_spike,
        CASE 
            WHEN daily_sentiment - LAG(daily_sentiment, 3) OVER (ORDER BY created_date) > 0.3 THEN 'MAJOR_IMPROVEMENT'
            WHEN daily_sentiment - LAG(daily_sentiment, 3) OVER (ORDER BY created_date) < -0.3 THEN 'MAJOR_DECLINE'
            ELSE 'STABLE'
        END as shift_type
    FROM daily_metrics
)
SELECT 
    created_date,
    ROUND(daily_sentiment, 3) as sentiment,
    feedback_volume,
    volume_spike,
    shift_type,
    CASE 
        WHEN volume_spike AND shift_type = 'MAJOR_DECLINE' THEN '🚨 HIGH ALERT'
        WHEN volume_spike OR shift_type IN ('MAJOR_DECLINE', 'MAJOR_IMPROVEMENT') THEN '⚠️ MONITOR'
        ELSE '✓ Normal'
    END as alert_level
FROM indicators
WHERE shift_type != 'STABLE' OR volume_spike = TRUE
ORDER BY created_date DESC
LIMIT 20;

---

## Paso 11: ML-Powered Sentiment Forecasting

### Qué Vamos a Hacer

Use **Snowflake ML FORECAST()** to train a time-series model and predict future sentiment trends.

### Why This Matters

- **True ML predictions**: Not just statistical trends, but learned patterns
- **Future visibility**: Predict sentiment 7-14 days ahead
- **Data-driven planning**: Make proactive decisions based on ML forecasts

### Technical Approach

1. **Prepare time-series data**: Daily sentiment averages
2. **Train ML model**: SNOWFLAKE.ML.FORECAST() on historical data
3. **Generar predictions**: Forecast future sentiment trajectory
4. **Store results**: Save predictions for dashboard and analysis

In [ ]:
-- Paso 1: Prepare time-series data for forecasting
-- NOTE: Only include timestamp and target columns (no exogenous features)
-- Additional columns would be treated as exogenous variables requiring future values
CREATE OR REPLACE VIEW sentiment_timeseries AS
SELECT 
    lsh.created_date as timestamp,
    ROUND(AVG(sf.analysis_result), 3) as sentiment_score
FROM launch_sentiment_history lsh
INNER JOIN sentiment_forecasts sf ON lsh.record_id = sf.record_id
GROUP BY lsh.created_date
ORDER BY lsh.created_date;

-- Paso 2: Train ML forecasting model
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST sentiment_forecast_model(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'sentiment_timeseries'),
    TIMESTAMP_COLNAME => 'TIMESTAMP',
    TARGET_COLNAME => 'SENTIMENT_SCORE'
);

-- Paso 3: Generar forecasts for next 14 days with 95% prediction interval
CALL sentiment_forecast_model!FORECAST(
    FORECASTING_PERIODS => 14,
    CONFIG_OBJECT => {'prediction_interval': 0.95}
);

-- Paso 4: View the ML-generated forecasts
SELECT 
    ts as forecast_date,
    ROUND(forecast, 3) as predicted_sentiment,
    ROUND(lower_bound, 3) as confidence_lower,
    ROUND(upper_bound, 3) as confidence_upper,
    DATEDIFF('day', CURRENT_DATE(), ts) as days_ahead,
    CASE 
        WHEN forecast >= 0.5 THEN '🟢 Positive Outlook'
        WHEN forecast >= 0 THEN '🟡 Neutral Outlook'
        ELSE '🔴 Negative Outlook'
    END as sentiment_outlook
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY ts
LIMIT 14;


---

## Paso 12: Dashboard Interactivo with ML Forecasts

### Dashboard Features

- **Overview Tab**: Key metrics, reception score, sentiment distribution
- **Forecast & Trends Tab**: Time-series charts, historical patterns, volatility
- **Predictions Tab**: Daily sentiment metrics with trend indicators
- **Details Tab**: Filterable feedback records, export functionality

In [ ]:
# Launch Reception Predictor & Sentiment Forecaster Dashboard - Enhanced with Altair
import streamlit as st
import pandas as pd
import altair as alt
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("🚀 Launch Reception Predictor & Sentiment Forecaster")
st.markdown("### Predict sentiment trajectory and reception for new product launches using ML")

# Create tabs for better organization
tab1, tab2, tab3, tab4 = st.tabs(["📊 Overview", "📈 Forecast & Trends", "🎯 Predictions", "🔍 Details"])

# ============================================================================
# TAB 1: OVERVIEW
# ============================================================================
with tab1:
    # Key metrics with enhanced styling
    col1, col2, col3, col4 = st.columns(4)
    
    try:
        total_records = session.sql("SELECT COUNT(*) as cnt FROM launch_sentiment_history").collect()[0]['CNT']
    except:
        total_records = 0
    
    try:
        avg_result = session.sql("SELECT ROUND(AVG(analysis_result), 3) as avg FROM sentiment_forecasts").collect()[0]['AVG']
        if avg_result is None:
            avg_result = 0.0
    except:
        avg_result = 0.0
    
    try:
        forecast_count = session.sql("SELECT COUNT(*) as cnt FROM launch_predictions").collect()[0]['CNT']
    except:
        forecast_count = 0
    
    try:
        confidence_avg = session.sql("SELECT ROUND(AVG(confidence), 3) as avg FROM sentiment_forecasts").collect()[0]['AVG']
        if confidence_avg is None:
            confidence_avg = 0.0
    except:
        confidence_avg = 0.0
    
    with col1:
        st.metric("Feedback Records", f"{total_records:,}", help="Total launch feedback analyzed")
    
    with col2:
        sentiment_emoji = "😊" if avg_result >= 0.5 else "😐" if avg_result >= -0.5 else "😟"
        st.metric("Avg Sentiment", f"{avg_result:.3f}", 
                  delta=f"{sentiment_emoji}",
                  delta_color="normal" if avg_result >= 0 else "inverse")
    
    with col3:
        st.metric("Forecasts Generated", forecast_count, help="ML-generated predictions")
    
    with col4:
        st.metric("Avg Confidence", f"{confidence_avg:.3f}", help="Model confidence score")
    
    # Launch Health Score with Progress Bar
    st.markdown("---")
    st.subheader("🎯 Launch Reception Score")
    
    launch_score = int((avg_result + 1) * 50)  # Convert -1 to 1 → 0 to 100
    
    col_left, col_right = st.columns([2, 1])
    
    with col_left:
        st.progress(launch_score / 100, text=f"Launch Score: {launch_score}/100")
        
        if launch_score >= 75:
            st.success("🟢 **Excellent!** Strong positive reception predicted")
        elif launch_score >= 50:
            st.info("👍 **Good!** Positive launch trajectory")
        elif launch_score >= 25:
            st.warning("⚠️ **Mixed** - Monitor closely and adjust strategy")
        else:
            st.error("🚨 **Concerning** - Intervention recommended")
    
    with col_right:
        # Sentiment distribution pie chart
        try:
            sentiment_breakdown = session.sql("""
                SELECT 
                    CASE 
                        WHEN sf.analysis_result >= 0.5 THEN 'Positive'
                        WHEN sf.analysis_result <= -0.5 THEN 'Negative'
                        ELSE 'Neutral'
                    END as sentiment_label,
                    COUNT(*) as count
                FROM sentiment_forecasts sf
                GROUP BY sentiment_label
            """).to_pandas()
            
            if not sentiment_breakdown.empty:
                pie_chart = alt.Chart(sentiment_breakdown).mark_arc(innerRadius=50).encode(
                    theta=alt.Theta(field="COUNT", type="quantitative"),
                    color=alt.Color(field="SENTIMENT_LABEL", type="nominal",
                                    scale=alt.Scale(
                                        domain=['Positive', 'Neutral', 'Negative'],
                                        range=['#00CC96', '#FFA15A', '#EF553B']
                                    ),
                                    legend=alt.Legend(title="Sentiment")),
                    tooltip=['SENTIMENT_LABEL', 'COUNT']
                ).properties(
                    width=200,
                    height=200,
                    title='Sentiment Distribution'
                )
                
                st.altair_chart(pie_chart, use_container_width=False)
        except:
            st.info("No sentiment distribution data available yet")

# ============================================================================
# TAB 2: FORECAST & TRENDS
# ============================================================================
with tab2:
    st.subheader("📈 Sentiment Forecast & Historical Trends")
    
    try:
        trend_data = session.sql("""
            SELECT 
                lsh.created_date,
                ROUND(AVG(sf.analysis_result), 3) as avg_sentiment,
                ROUND(AVG(sf.confidence), 3) as avg_confidence,
                COUNT(*) as feedback_count
            FROM launch_sentiment_history lsh
            INNER JOIN sentiment_forecasts sf ON lsh.record_id = sf.record_id
            GROUP BY lsh.created_date
            ORDER BY lsh.created_date
        """).to_pandas()
        
        if not trend_data.empty:
            # Dual-axis chart: sentiment line + confidence
            base = alt.Chart(trend_data).encode(
                x=alt.X('CREATED_DATE:T', title='Date', axis=alt.Axis(format='%b %d'))
            )
            
            # Sentiment line
            sentiment_line = base.mark_line(point=True, strokeWidth=3, color='#636EFA').encode(
                y=alt.Y('AVG_SENTIMENT:Q', title='Avg Sentiment', scale=alt.Scale(domain=[-1, 1])),
                tooltip=[
                    alt.Tooltip('CREATED_DATE:T', title='Date', format='%Y-%m-%d'),
                    alt.Tooltip('AVG_SENTIMENT:Q', title='Sentiment', format='.3f'),
                    alt.Tooltip('FEEDBACK_COUNT:Q', title='Feedback Count')
                ]
            )
            
            # Threshold lines
            positive_line = alt.Chart(pd.DataFrame({'y': [0.5]})).mark_rule(
                strokeDash=[5, 5], color='green', opacity=0.5
            ).encode(y='y:Q')
            
            negative_line = alt.Chart(pd.DataFrame({'y': [-0.5]})).mark_rule(
                strokeDash=[5, 5], color='red', opacity=0.5
            ).encode(y='y:Q')
            
            zero_line = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(
                strokeDash=[2, 2], color='gray', opacity=0.7
            ).encode(y='y:Q')
            
            final_chart = (sentiment_line + positive_line + negative_line + zero_line).properties(
                width=800,
                height=400,
                title='Launch Sentiment Trajectory'
            )
            
            st.altair_chart(final_chart, use_container_width=True)
            
            # Trend statistics
            col1, col2, col3, col4 = st.columns(4)
            
            with col1:
                recent_avg = trend_data['AVG_SENTIMENT'].tail(7).mean()
                st.metric("Recent 7-Day Avg", f"{recent_avg:.3f}")
            
            with col2:
                peak = trend_data['AVG_SENTIMENT'].max()
                st.metric("Peak Sentiment", f"{peak:.3f}")
            
            with col3:
                valley = trend_data['AVG_SENTIMENT'].min()
                st.metric("Lowest Sentiment", f"{valley:.3f}")
            
            with col4:
                volatility = trend_data['AVG_SENTIMENT'].std()
                st.metric("Volatility", f"{volatility:.3f}")
        else:
            st.info("No trend data available yet")
    except Exception as e:
        st.warning(f"Trend data not available: {str(e)}")

# ============================================================================
# TAB 3: PREDICTIONS
# ============================================================================
with tab3:
    st.subheader("🎯 Daily Sentiment Metrics & Trends")
    
    try:
        # Query actual table schema: insight_id, date_period, metric_name, metric_value, trend, calculated_at
        metrics = session.sql("""
            SELECT 
                date_period,
                metric_value,
                trend
            FROM launch_predictions
            ORDER BY date_period DESC
            LIMIT 30
        """).to_pandas()
        
        if not metrics.empty:
            # Display as interactive chart
            st.markdown("### 📈 Daily Sentiment Trend")
            
            trend_chart = alt.Chart(metrics).mark_line(point=True, strokeWidth=2).encode(
                x=alt.X('DATE_PERIOD:T', title='Date', axis=alt.Axis(format='%b %d')),
                y=alt.Y('METRIC_VALUE:Q', title='Daily Avg Sentiment', scale=alt.Scale(domain=[-1, 1])),
                color=alt.Color('TREND:N', 
                               scale=alt.Scale(
                                   domain=['up', 'stable', 'down'],
                                   range=['green', 'gray', 'red']
                               ),
                               legend=alt.Legend(title='Trend')),
                tooltip=[
                    alt.Tooltip('DATE_PERIOD:T', title='Date', format='%Y-%m-%d'),
                    alt.Tooltip('METRIC_VALUE:Q', title='Sentiment', format='.3f'),
                    alt.Tooltip('TREND:N', title='Trend')
                ]
            ).properties(
                width=700,
                height=400,
                title='Daily Average Sentiment with Trend Indicators'
            )
            
            st.altair_chart(trend_chart, use_container_width=True)
            
            # Summary statistics
            col1, col2, col3, col4 = st.columns(4)
            
            with col1:
                recent_avg = metrics['METRIC_VALUE'].head(7).mean()
                st.metric("Recent 7-Day Avg", f"{recent_avg:.3f}")
            
            with col2:
                up_days = (metrics['TREND'] == 'up').sum()
                st.metric("Improving Days", up_days)
            
            with col3:
                down_days = (metrics['TREND'] == 'down').sum()
                st.metric("Declining Days", down_days)
            
            with col4:
                latest = metrics.iloc[0]['METRIC_VALUE']
                st.metric("Latest Sentiment", f"{latest:.3f}")
            
            # Detailed table
            st.markdown("### 📊 Detailed Metrics")
            st.dataframe(
                metrics,
                use_container_width=True,
                hide_index=True,
                column_config={
                    "DATE_PERIOD": st.column_config.DateColumn("Date", format="YYYY-MM-DD"),
                    "METRIC_VALUE": st.column_config.NumberColumn("Sentiment", format="%.3f"),
                    "TREND": st.column_config.TextColumn("Trend")
                }
            )
            
            # Download button
            csv = metrics.to_csv(index=False)
            st.download_button(
                label="📥 Download Metrics",
                data=csv,
                file_name="sentiment_metrics.csv",
                mime="text/csv"
            )
        else:
            st.info("No metrics available yet. Run cells 1-5 to generate sentiment metrics.")
    except Exception as e:
        st.warning(f"Metrics data not available: {str(e)}")

# ============================================================================
# TAB 4: DETAILS
# ============================================================================
with tab4:
    st.subheader("🔍 Detailed Feedback Analysis")
    
    # Filters
    col1, col2 = st.columns(2)
    
    with col1:
        sentiment_filter = st.selectbox(
            "Filter by Sentiment",
            ["All", "Positive (≥ 0.5)", "Neutral (-0.5 to 0.5)", "Negative (≤ -0.5)"]
        )
    
    with col2:
        limit = st.slider("Number of records", 10, 100, 50)
    
    # Build query based on filter
    if sentiment_filter == "Positive (≥ 0.5)":
        where_clause = "WHERE sf.analysis_result >= 0.5"
    elif sentiment_filter == "Negative (≤ -0.5)":
        where_clause = "WHERE sf.analysis_result <= -0.5"
    elif sentiment_filter == "Neutral (-0.5 to 0.5)":
        where_clause = "WHERE sf.analysis_result > -0.5 AND sf.analysis_result < 0.5"
    else:
        where_clause = ""
    
    try:
        recent_data = session.sql(f"""
            SELECT 
                lsh.created_date,
                ROUND(sf.analysis_result, 3) as sentiment_score,
                CASE 
                    WHEN sf.analysis_result >= 0.5 THEN '🟢 Positive'
                    WHEN sf.analysis_result <= -0.5 THEN '🔴 Negative'
                    ELSE '🟡 Neutral'
                END as sentiment_label,
                sf.confidence
            FROM launch_sentiment_history lsh
            INNER JOIN sentiment_forecasts sf ON lsh.record_id = sf.record_id
            {where_clause}
            ORDER BY lsh.created_date DESC, sf.confidence DESC
            LIMIT {limit}
        """).to_pandas()
        
        if not recent_data.empty:
            st.dataframe(
                recent_data,
                use_container_width=True,
                hide_index=True
            )
            
            # Download button
            csv = recent_data.to_csv(index=False)
            st.download_button(
                label="📥 Download Feedback Data",
                data=csv,
                file_name="launch_feedback.csv",
                mime="text/csv"
            )
        else:
            st.info("No records match the selected filters.")
    except Exception as e:
        st.error(f"Error loading data: {str(e)}")

# ============================================================================
# FOOTER
# ============================================================================
st.markdown("---")
st.success("✅ Launch predictor operational | Forecasts current")

# Info section
with st.expander("ℹ️ About This Dashboard"):
    st.markdown("""
    ### Launch Reception Predictor System
    
    **Overview Tab:**
    - Key launch metrics and reception score
    - Sentiment distribution visualization
    
    **Forecast & Trends Tab:**
    - Time-series sentiment trajectory
    - Historical trends with predictions
    - Volatility analysis
    
    **Predictions Tab:**
    - ML-generated launch reception forecasts
    - Confidence-weighted predictions
    - Strategic recommendations
    
    **Details Tab:**
    - Filterable feedback records
    - Export functionality
    
    **ML Features Used:**
    - `AI_SENTIMENT()` for text analysis
    - `SNOWFLAKE.ML.FORECAST()` for prediction (if used)
    - Statistical trend analysis
    
    **Sentiment Scoring:**
    - **Positive**: ≥ 0.5
    - **Neutral**: -0.5 to 0.5
    - **Negative**: ≤ -0.5
    """)

---

##  Tutorial Complete!

### What You've Learned

1.  **AI-powered sentiment analysis** with Snowflake Cortex SENTIMENT()
2.  **ML-powered forecasting** with SNOWFLAKE.ML.FORECAST()
3.  **Production data pipelines** with SQL
4.  **Time-series analysis** with window functions
5.  **Statistical trend detection** with aggregations
6.  **Interactive dashboards** with Streamlit

### Key Snowflake Features Used

- **`AI_SENTIMENT()`** - AI sentiment analysis
- **`SNOWFLAKE.ML.FORECAST()`** - ML time-series forecasting
- **Window functions** - Rolling averages and trends
- **CTEs** - Complex query logic
- **Aggregations** - Summary statistics
- **Streamlit** - Interactive visualization

### Next Steps for Production

1. **Connect real data sources** - Replace synthetic data
2. **Automate data refresh** - Schedule regular updates
3. **Retrain forecast model** - Update with new data
4. **Add alerting logic** - Notify on prediction changes
5. **Scale to production volumes** - Handle millions of records
6. **Integrate with reporting tools** - Export to BI platforms

---

**Congratulations!** You've built a production-ready launch reception predictor using **Snowflake Cortex AI** (SENTIMENT) and **Snowflake ML** (FORECAST).

**Estimated runtime**: (varies by data size)

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `LAUNCH_PREDICTOR_DB` database (and all tables/models within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data and models will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS LAUNCH_PREDICTOR_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;
